# Pandas: Combining & Grouping Data
### Topics Covered (in order)
1. **Concatenation** — Stack DataFrames simply
2. **Merging** — Combine using matching column values
3. **Joining** — Similar to merge, index-based
4. **Ranking** — Assign numeric ranks to data
5. **Grouping** — Split, aggregate, and summarise

---

## ⚙️ Step 0 — Install Required Libraries
Run this cell once. These packages are needed for the entire notebook.

In [ ]:
# Install pandas — the main data manipulation library we'll use throughout
!pip install pandas

# Install numpy — used for numerical operations and NaN values
!pip install numpy

## 📦 Step 1 — Import Libraries & Load Data

In [ ]:
# Import pandas with the standard alias 'pd'
import pandas as pd

# Import numpy with the standard alias 'np' — we'll use np.nan for missing values
import numpy as np

print("Libraries loaded successfully!")

In [ ]:
# Load the sales CSV file — index_col sets Date as the row label, parse_dates converts it to datetime
df = pd.read_csv('data/sales.csv', index_col='Date', parse_dates=True)

# Preview the first 5 rows to confirm it loaded correctly
df.head()

**What's in our data?**
- `Rep` — salesperson name
- `Sector` — industry sector
- `Units` — number of units sold
- `Revenue` — money earned

---

---

# 📌 TOPIC 1 — Concatenation

> **What it does:** Simply *stacks* DataFrames together — like gluing pages of a book.
> No intelligence. No matching. Just stack.

**Use it when:** You have the same columns in multiple DataFrames and just want to combine them.

In [ ]:
# Create three small DataFrames — each represents one month of data
# January: rows where the month is 2024-01
df_jan = df['2024-01'].copy()

# February data
df_feb = df['2024-02'].copy()

# March data
df_mar = df['2024-03'].copy()

print("January:")
display(df_jan)
print("\nFebruary:")
display(df_feb)

### 1A — Concatenate Rows (stack vertically)
This is the **default** — rows are stacked one below another.

In [ ]:
# pd.concat takes a LIST of DataFrames and stacks them vertically (by rows) by default
df_all_months = pd.concat([df_jan, df_feb, df_mar])

# Display the result — notice all rows from all three months are now in one DataFrame
print(f"Total rows after concat: {len(df_all_months)}")
display(df_all_months)

### 1B — Concatenate Columns (stack side by side)
Use `axis=1` to place DataFrames **side by side** (add new columns).

In [ ]:
# Create two DataFrames with DIFFERENT columns but same index (dates)
# First DataFrame: only Units column
df_units = df[['Units']].copy()

# Second DataFrame: only Revenue column
df_revenue = df[['Revenue']].copy()

# axis=1 means "stack side by side" (along columns, not rows)
df_side_by_side = pd.concat([df_units, df_revenue], axis=1)

print("Units and Revenue side by side:")
display(df_side_by_side)

### ⚠️ What happens when dimensions don't match?
If you concat along rows but columns differ, pandas fills missing spots with **NaN**.

In [ ]:
# df_a has columns A and B
df_a = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})

# df_b only has column B and C — column A is missing
df_b = pd.DataFrame({'B': [5, 6], 'C': [7, 8]})

# Concatenating these: A and C have no match in the other, so NaN appears
pd.concat([df_a, df_b])

**Key takeaway:** concat just stacks — it doesn't try to align values intelligently. For that, we use **merge**.

---

---

# 📌 TOPIC 2 — Merging

> **What it does:** Combines two DataFrames by matching values in a chosen column (like a JOIN in SQL).

**Use it when:** You have two separate tables sharing a common column (like an ID or a date).

There are **4 types** of merge — let's build two small tables first to see each one clearly.

In [ ]:
# Left table: Units sold by Alice and Bob
df_left = pd.DataFrame({
    'Rep':   ['Alice', 'Bob', 'Charlie'],
    'Units': [120,     95,    150]
})

# Right table: Targets for Bob, Charlie, and Diana (note: Alice is missing, Diana is extra)
df_right = pd.DataFrame({
    'Rep':    ['Bob', 'Charlie', 'Diana'],
    'Target': [100,   140,       80]
})

print("Left DataFrame:")
display(df_left)
print("\nRight DataFrame:")
display(df_right)

### 2A — Inner Merge
**Keep only rows where the key exists in BOTH DataFrames.**
Alice (only in left) and Diana (only in right) are dropped.

In [ ]:
# how='inner' is actually the DEFAULT — only matching rows on both sides survive
# on='Rep' tells pandas which column to match on
df_inner = pd.merge(df_left, df_right, how='inner', on='Rep')

# Result: only Bob and Charlie appear (they exist in both tables)
display(df_inner)

### 2B — Left Merge
**Keep ALL rows from the LEFT DataFrame.** Rows from the right are matched where possible; otherwise NaN is used.

In [ ]:
# how='left' — every row in df_left is kept
# Alice has no match in df_right, so Target becomes NaN
# Diana is in df_right but not df_left, so she is excluded
df_left_merge = pd.merge(df_left, df_right, how='left', on='Rep')

display(df_left_merge)

### 2C — Right Merge
**Keep ALL rows from the RIGHT DataFrame.** Rows from the left are matched where possible; otherwise NaN.

In [ ]:
# how='right' — every row in df_right is kept
# Diana has no match in df_left, so Units becomes NaN
# Alice is in df_left but not df_right, so she is excluded
df_right_merge = pd.merge(df_left, df_right, how='right', on='Rep')

display(df_right_merge)

### 2D — Outer Merge
**Keep ALL rows from BOTH DataFrames.** Fill with NaN wherever a value is missing.

In [ ]:
# how='outer' — nothing is dropped. Alice gets NaN for Target, Diana gets NaN for Units
df_outer = pd.merge(df_left, df_right, how='outer', on='Rep')

display(df_outer)

### 2E — The `suffixes` parameter
When both DataFrames have a column with the **same name** (other than the merge key), pandas adds `_x` and `_y` by default. You can customise these with `suffixes`.

In [ ]:
# Both tables have a 'Score' column — pandas needs to differentiate them
df_q1 = pd.DataFrame({'Rep': ['Alice', 'Bob'], 'Score': [88, 74]})
df_q2 = pd.DataFrame({'Rep': ['Alice', 'Bob'], 'Score': [91, 80]})

# Without suffixes: pandas auto-adds _x and _y
print("Without custom suffixes:")
display(pd.merge(df_q1, df_q2, on='Rep'))

# With suffixes: we give meaningful names to each side
print("\nWith custom suffixes:")
display(pd.merge(df_q1, df_q2, on='Rep', suffixes=('_Q1', '_Q2')))

### 2F — Merging on the Index (`left_index` / `right_index`)
Sometimes the column you want to merge on is the **index**, not a regular column. Use `left_index=True` or `right_index=True`.

In [ ]:
# Create two DataFrames where Rep is the INDEX (not a column)
df_units_idx = pd.DataFrame({'Units': [120, 95]}, index=['Alice', 'Bob'])
df_target_idx = pd.DataFrame({'Target': [130, 100]}, index=['Alice', 'Bob'])

# left_index=True means "use the left DataFrame's index as the merge key"
# right_index=True means "use the right DataFrame's index as the merge key"
pd.merge(df_units_idx, df_target_idx, how='inner', left_index=True, right_index=True)

---

---

# 📌 TOPIC 3 — Joining

> **What it does:** Almost identical to merge, but the `on` column **must be the index** in at least one DataFrame.

| | `pd.merge()` | `df.join()` |
|---|---|---|
| Call style | `pd.merge(df1, df2, ...)` | `df1.join(df2, ...)` |
| `on` can be | any column or index | must be index in at least one |
| Types | inner, left, right, outer | inner, left, right, outer |


In [ ]:
# Create two DataFrames using the date index from our sales data
# df_units: only the Units column for Jan
df_units = df[['Units']]['2024-01'].copy()

# df_rev: only the Revenue column — but for a slightly different date range (late-Jan + Feb)
df_rev = df[['Revenue']]['2024-01-19':'2024-02-09'].copy()

print("df_units (Jan):")
display(df_units)
print("\ndf_rev (late Jan + Feb):")
display(df_rev)

### 3A — Default Join (Left)
By default, `.join()` keeps all rows of the **left** DataFrame (same as a left merge).

In [ ]:
# Default join — keeps all rows from df_units (left)
# Revenue is NaN for any date not present in df_rev
df_units.join(df_rev)

### 3B — Inner Join
Only keep dates that appear in **both** DataFrames.

In [ ]:
# how='inner' — only rows where the date index exists in BOTH DataFrames
df_units.join(df_rev, how='inner')

### 3C — Right Join
Keep all rows from the **right** DataFrame.

In [ ]:
# how='right' — all rows from df_rev (right) are kept
# Units is NaN for dates not in df_units
df_units.join(df_rev, how='right')

### 3D — Outer Join
Keep **everything** from both — fill gaps with NaN.

In [ ]:
# how='outer' — all dates from both DataFrames are included, NaN where data is missing
df_units.join(df_rev, how='outer')

---

---

# 📌 TOPIC 4 — Ranking Data

> **What it does:** Assigns a numerical rank (1st, 2nd, 3rd...) to each row based on a column's value.

- Use the `.rank()` method on a Series or DataFrame column
- In case of a **tie**, the default is to assign the **average** rank to both

In [ ]:
# Work with a copy of the full DataFrame
df_rank = df.copy()

# Add a new column 'Rank' — each row gets a rank based on its Units value
# The row with the fewest units gets rank 1, the most gets rank 12 (since we have 12 rows)
df_rank['Rank'] = df_rank['Units'].rank()

df_rank

### 4A — Ascending Rank (default)
The **lowest value** gets rank 1.

In [ ]:
# Sort the DataFrame by rank to confirm lowest Units = rank 1
# ascending=True is the DEFAULT — smallest value → rank 1
df_rank['Rank'] = df_rank['Units'].rank(ascending=True)

# Sort by Rank column to make it easy to read
df_rank.sort_values(by='Rank')

### 4B — Descending Rank
The **highest value** gets rank 1 — useful for leaderboards.

In [ ]:
# ascending=False flips the ranking — the HIGHEST value now gets rank 1
df_rank['Rank'] = df_rank['Units'].rank(ascending=False)

# Sort by Rank — now the row with most units should be at the top
df_rank.sort_values(by='Rank')

### 4C — Tie Handling
When two rows have the same value, the default is to assign them the **average** of the ranks they would have occupied.

In [ ]:
# Create a simple DataFrame with tied values
df_ties = pd.DataFrame({'Score': [90, 90, 80, 70, 70]})

# Add a rank column — notice what happens with the two 90s and the two 70s
# The two 90s would be ranks 4 and 5, so they both get (4+5)/2 = 4.5
# The two 70s would be ranks 1 and 2, so they both get (1+2)/2 = 1.5
df_ties['Rank'] = df_ties['Score'].rank()

df_ties

### 4D — Ranking with a Filter
A common real-world use: rank only the rows that match a condition.

In [ ]:
# Filter: only look at rows where Revenue is greater than 18000
high_rev_filter = df['Revenue'] > 18000

# Apply the filter to get only those rows, then rank the Units column
df_filtered = df[high_rev_filter].copy()

# Rank the Units within this filtered set (ascending=False → top performers get rank 1)
df_filtered['Rank'] = df_filtered['Units'].rank(ascending=False)

# Sort by rank to show the leaderboard
df_filtered.sort_values(by='Rank')

---

---

# 📌 TOPIC 5 — Grouping Data

> **What it does:** Splits the DataFrame into groups, applies a function (like mean, sum, std) to each group, and combines the results.

Think of it like: *"Give me the average revenue per Sector"* or *"Total units sold per Rep per month"*.

The method is `.groupby()`. After grouping, you call an **aggregate function**: `mean()`, `sum()`, `std()`, `agg()`, etc.

### 5A — Group by a Single Column

In [ ]:
# Group rows by the 'Sector' column, then compute the mean for each group
# This answers: "What is the average Units and Revenue per Sector?"
df.groupby(by='Sector').mean(numeric_only=True)

In [ ]:
# Group by 'Rep' instead — "What is the average performance per salesperson?"
df.groupby(by='Rep').mean(numeric_only=True)

In [ ]:
# We can also use .agg() and pass the function name as a string
# This makes it easy to swap out functions dynamically
func = 'std'  # standard deviation — how spread out the numbers are

df.groupby(by='Sector').agg(func, numeric_only=True)

### 5B — Group by Multiple Columns
You can group by more than one column by passing a list.

In [ ]:
# Group by both Sector AND Rep — each unique combination becomes a group
# This answers: "What is the average revenue for each Rep within each Sector?"
df.groupby(by=['Sector', 'Rep']).mean(numeric_only=True)

### 5C — Group by Time Period
When your index is a **datetime**, you can group by time frequencies like month, quarter, or year.

Use `pd.Grouper(freq=...)` — it's designed exactly for this.

In [ ]:
# pd.Grouper(freq='ME') groups by Month End
# Common frequencies: 'ME' (month end), 'QE' (quarter end), 'YE' (year end), 'W' (week), 'B' (business day)
df.groupby(pd.Grouper(freq='ME')).mean(numeric_only=True)

In [ ]:
# Store the Grouper in a variable for readability — same result but easier to reuse
by_month = pd.Grouper(freq='ME')

# Now use agg() with a function name as a string
func = 'sum'  # Total units and revenue per month

df.groupby(by=by_month).agg(func, numeric_only=True)

### 5D — Multi-level Grouping (combining time + columns)
You can combine a `pd.Grouper` (for time) with regular column groupers in a list for powerful slice-and-dice operations.

In [ ]:
# Create Grouper objects — one for time (month), one for each category column
by_month  = pd.Grouper(freq='ME')          # groups by calendar month
by_rep    = pd.Grouper(key='Rep')           # groups by salesperson
by_sector = pd.Grouper(key='Sector')        # groups by industry sector

# Combine: group by Sector first, then Rep, then Month
# This answers: "For each Sector > Rep > Month, what is the mean revenue?"
groups1 = [by_sector, by_rep, by_month]

df.groupby(by=groups1).agg('mean', numeric_only=True)

In [ ]:
# Change the order of groupers to get a different slice of the data
# Now: Month first, then Sector, then Rep
groups2 = [by_month, by_sector, by_rep]

df.groupby(by=groups2).agg('mean', numeric_only=True)

---

# ✅ Summary — What We Covered

| Topic | Method | When to use |
|---|---|---|
| **Concatenation** | `pd.concat([df1, df2])` | Simple stacking — same columns, just more rows (or same rows, more columns with `axis=1`) |
| **Merging** | `pd.merge(df1, df2, how=..., on=...)` | Combining on a shared column value; 4 types: inner, left, right, outer |
| **Joining** | `df1.join(df2, how=...)` | Like merge, but the key must be the index; cleaner syntax for index-based joins |
| **Ranking** | `df['col'].rank(ascending=...)` | Assign numeric positions to rows; useful for leaderboards and comparisons |
| **Grouping** | `df.groupby(by=...).agg(...)` | Summarise data by categories or time periods using aggregate functions |

---
*End of notebook*